# wandb Sweep 결과 조회 예시

저장된 데이터:
- **Tables**: `conversation`, `query_transforms`, `sessions_detail`, `books_detail`
- **Metrics**: `match_rate/*`, `token/*`, `verdict_code`
- **Config**: `persona_name`, `query_transform`, `judge_model`, `collection_name`

In [ ]:
import pandas as pd
import wandb

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

ENTITY  = "jjeong3150-aiffel"
PROJECT = "peekabook-crs-multisession-test1"

api = wandb.Api()

## 1. 전체 runs 목록 조회

In [ ]:
def list_runs(
    persona_name:    str | None = None,
    query_transform: str | None = None,
    judge_model:     str | None = None,
    collection_name: str | None = None,
) -> pd.DataFrame:
    """Sweep 결과 runs를 필터링해 DataFrame으로 반환."""
    filters: dict = {}
    if persona_name:
        filters["config.persona_name"] = persona_name
    if query_transform:
        filters["config.query_transform"] = query_transform
    if judge_model:
        filters["config.judge_model"] = judge_model
    if collection_name:
        filters["config.collection_name"] = collection_name

    runs = api.runs(f"{ENTITY}/{PROJECT}", filters=filters)

    rows = []
    for run in runs:
        rows.append({
            "run_id":             run.id,
            "run_name":           run.name,
            "state":              run.state,
            "persona_name":       run.config.get("persona_name"),
            "query_transform":    run.config.get("query_transform"),
            "judge_model":        run.config.get("judge_model"),
            "collection_name":    run.config.get("collection_name"),
            "n_sessions":         run.config.get("n_sessions"),
            "judge_match_rate":   run.summary.get("match_rate/peekajudge"),
            "self_match_rate":    run.summary.get("match_rate/peekareader_self"),
            "token_total_input":  run.summary.get("token/total_input"),
            "token_total_output": run.summary.get("token/total_output"),
            "token_crs_input":    run.summary.get("token/crs_input"),
            "token_judge_input":  run.summary.get("token/judge_input"),
        })

    return pd.DataFrame(rows)

In [ ]:
df_runs = list_runs()
df_runs

## 2. 특정 run의 Table 데이터 조회

저장된 Tables: `conversation`, `query_transforms`, `sessions_detail`, `books_detail`

In [ ]:
def _get_table(run: wandb.apis.public.Run, table_key: str) -> pd.DataFrame:
    """run summary 또는 artifacts에서 wandb.Table을 DataFrame으로 변환."""
    # summary에서 직접 읽기
    summary_table = run.summary.get(table_key)
    if summary_table and isinstance(summary_table, dict):
        cols = summary_table.get("columns", [])
        data = summary_table.get("data", [])
        return pd.DataFrame(data, columns=cols)

    # artifacts에서 읽기
    for artifact in run.logged_artifacts():
        table = artifact.get(table_key)
        if table is not None:
            return table.get_dataframe()

    return pd.DataFrame()


def get_conversation(run_id: str) -> pd.DataFrame:
    """대화 내역: persona_name, session_id, turn, crs, thought, user."""
    run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")
    return _get_table(run, "conversation")


def get_query_transforms(run_id: str) -> pd.DataFrame:
    """쿼리 변환 내역: original, step_back, rewritten, sub_queries, hypothetical_doc, all_queries."""
    run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")
    return _get_table(run, "query_transforms")


def get_sessions_detail(run_id: str) -> pd.DataFrame:
    """세션별 요약: session_id, preferred_genre, status, self_match_rate, judge_match_rate, verdict."""
    run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")
    return _get_table(run, "sessions_detail")


def get_books_detail(run_id: str) -> pd.DataFrame:
    """추천 도서별 Judge 결과: rank, title, author, judge_score, judge_reason."""
    run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")
    return _get_table(run, "books_detail")

In [ ]:
# 조회할 run_id 지정 (df_runs에서 선택하거나 직접 입력)
SAMPLE_RUN_ID = df_runs.iloc[0]["run_id"] if not df_runs.empty else "여기에_run_id_입력"
print(f"조회 대상 run: {SAMPLE_RUN_ID}")

In [ ]:
# 대화 내역
df_conv = get_conversation(SAMPLE_RUN_ID)
df_conv

In [ ]:
# 쿼리 변환 내역
df_qt = get_query_transforms(SAMPLE_RUN_ID)
df_qt

In [ ]:
# 세션별 요약
df_sessions = get_sessions_detail(SAMPLE_RUN_ID)
df_sessions

In [ ]:
# 추천 도서별 Judge 결과
df_books = get_books_detail(SAMPLE_RUN_ID)
df_books

## 3. query_transform별 judge match_rate 비교

In [ ]:
def compare_query_transforms(
    persona_name:    str | None = None,
    collection_name: str | None = None,
) -> pd.DataFrame:
    """query_transform 종류별 평균 judge match rate와 토큰 비교."""
    df = list_runs(persona_name=persona_name, collection_name=collection_name)
    if df.empty:
        return df
    return (
        df.groupby("query_transform")
        .agg(
            n_runs=("run_id", "count"),
            judge_match_rate_mean=("judge_match_rate", "mean"),
            self_match_rate_mean=("self_match_rate", "mean"),
            token_total_input_mean=("token_total_input", "mean"),
            token_total_output_mean=("token_total_output", "mean"),
        )
        .round(4)
        .sort_values("judge_match_rate_mean", ascending=False)
    )

In [ ]:
compare_query_transforms()

## 4. 모델별 비교

In [ ]:
def compare_judge_models(persona_name: str | None = None) -> pd.DataFrame:
    """judge_model별 평균 match rate 비교."""
    df = list_runs(persona_name=persona_name)
    if df.empty:
        return df
    return (
        df.groupby("judge_model")
        .agg(
            n_runs=("run_id", "count"),
            judge_match_rate_mean=("judge_match_rate", "mean"),
            token_judge_input_mean=("token_judge_input", "mean"),
        )
        .round(4)
        .sort_values("judge_match_rate_mean", ascending=False)
    )

In [ ]:
compare_judge_models()

## 5. 페르소나별 비교

In [ ]:
def compare_personas(query_transform: str | None = None) -> pd.DataFrame:
    """페르소나별 평균 judge match rate 비교."""
    df = list_runs(query_transform=query_transform)
    if df.empty:
        return df
    return (
        df.groupby("persona_name")
        .agg(
            n_runs=("run_id", "count"),
            judge_match_rate_mean=("judge_match_rate", "mean"),
            self_match_rate_mean=("self_match_rate", "mean"),
        )
        .round(4)
        .sort_values("judge_match_rate_mean", ascending=False)
    )

In [ ]:
compare_personas()

## 6. 토큰 사용량 상세

In [ ]:
def get_token_summary(query_transform: str | None = None) -> pd.DataFrame:
    """CRS / UserSim / Judge 구성요소별 토큰 집계."""
    filters = {"config.query_transform": query_transform} if query_transform else {}
    runs = api.runs(f"{ENTITY}/{PROJECT}", filters=filters)
    rows = []
    for run in runs:
        s = run.summary
        rows.append({
            "run_name":        run.name,
            "query_transform": run.config.get("query_transform"),
            "persona_name":    run.config.get("persona_name"),
            "crs_input":       s.get("token/crs_input", 0),
            "crs_output":      s.get("token/crs_output", 0),
            "usersim_input":   s.get("token/usersim_input", 0),
            "usersim_output":  s.get("token/usersim_output", 0),
            "judge_input":     s.get("token/judge_input", 0),
            "judge_output":    s.get("token/judge_output", 0),
            "total_input":     s.get("token/total_input", 0),
            "total_output":    s.get("token/total_output", 0),
        })
    return pd.DataFrame(rows)

In [ ]:
df_tokens = get_token_summary()
df_tokens

In [ ]:
# query_transform별 토큰 합계
df_tokens.groupby("query_transform")[["crs_input", "usersim_input", "judge_input", "total_input"]].mean().round(0)